# ML-07 - Baseline Action Score and Top-10 Review

This notebook does the three assignment tasks in one place: check two signals first, encode one transparent rule, and review the top ten rows by hand.

## 1. Two signal checks first, then the rule

Plain-words rule idea: start from the CTR-fix story, not the trend label story. If a page is already visible in positions 1-20 and its CTR is below the weighted CTR usually seen in that position bucket, it gets a higher score. I tested staleness first because FlyRank refresh flags lean on it, but I only keep signals that survive the bucket checks below.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

csv_path = Path("D:/Flyrank/Flyrank-assignments/data/raw/content_refresh_anonymized.csv")
df = pd.read_csv(csv_path)
visible = df[(df["impressions_90d"] >= 1000) & (df["avg_position"] > 0)].copy()

visible["position_bucket"] = pd.cut(
    visible["avg_position"],
    bins=[0, 3, 10, 20, 50, 200],
    labels=["top_3", "page_1_not_top_3", "page_2", "page_3_5", "deep"],
)
position_signal = (
    visible.groupby("position_bucket", observed=False)
    .agg(
        n=("content_id", "size"),
        total_clicks=("clicks_90d", "sum"),
        total_impressions=("impressions_90d", "sum"),
        median_ctr_pct=("ctr", "median"),
    )
    .reset_index()
)
position_signal["weighted_ctr_pct"] = (
    position_signal["total_clicks"] / position_signal["total_impressions"] * 100
).round(3)
position_signal["median_ctr_pct"] = position_signal["median_ctr_pct"].round(3)
position_signal = position_signal[["position_bucket", "n", "weighted_ctr_pct", "median_ctr_pct"]]

flag_slice = visible[visible["avg_position"].between(1, 20)].copy()
flag_slice["staleness_bucket"] = np.where(
    flag_slice["days_since_last_update"] >= 91,
    "91_plus_days",
    "0_90_days",
)
staleness_signal = (
    flag_slice.groupby("staleness_bucket")
    .agg(
        n=("content_id", "size"),
        total_clicks=("clicks_90d", "sum"),
        total_impressions=("impressions_90d", "sum"),
        median_ctr_pct=("ctr", "median"),
        median_position=("avg_position", "median"),
    )
    .reset_index()
)
staleness_signal["weighted_ctr_pct"] = (
    staleness_signal["total_clicks"] / staleness_signal["total_impressions"] * 100
).round(3)
staleness_signal["median_ctr_pct"] = staleness_signal["median_ctr_pct"].round(3)
staleness_signal["median_position"] = staleness_signal["median_position"].round(1)
staleness_signal = staleness_signal[["staleness_bucket", "n", "weighted_ctr_pct", "median_ctr_pct", "median_position"]]

signal_verdicts = pd.DataFrame(
    [
        {
            "signal": "CTR vs position bucket",
            "verdict": "CONFIRMED",
            "why": "Weighted CTR falls sharply as rank gets worse, so a CTR-fix rule should focus on already-visible pages.",
        },
        {
            "signal": "Staleness behind refresh flags",
            "verdict": "OPPOSITE",
            "why": "Inside positions 1-20, 91+ day pages do not have worse CTR than 0-90 day pages, so staleness should not drive this CTR rule.",
        },
    ]
)

print("Signal 1 - CTR vs position bucket (n shown per bucket)")
print(position_signal.to_string(index=False))
print("\nSignal 2 - staleness behind refresh flags, checked only on workable positions 1-20 (n shown per bucket)")
print(staleness_signal.to_string(index=False))
print("\nOne-word verdicts")
print(signal_verdicts.to_string(index=False))
print("\nFinal rule after the checks: score the CTR gap against the page's position bucket, scale it by visibility, and do not add a staleness bonus because that signal failed here.")

Signal 1 - CTR vs position bucket (n shown per bucket)
 position_bucket    n  weighted_ctr_pct  median_ctr_pct
           top_3  416             0.491           0.235
page_1_not_top_3 6175             0.349           0.240
          page_2 3420             0.356           0.190
        page_3_5 3312             0.155           0.090
            deep  189             0.036           0.000

Signal 2 - staleness behind refresh flags, checked only on workable positions 1-20 (n shown per bucket)
staleness_bucket    n  weighted_ctr_pct  median_ctr_pct  median_position
       0_90_days 6328             0.356            0.23              8.1
    91_plus_days 3671             0.365            0.22              7.3

One-word verdicts
                        signal   verdict                                                                                                                          why
        CTR vs position bucket CONFIRMED                      Weighted CTR falls sharply as rank get

## 2. Build the ranked queue and write the CSV

The final rule uses only current-window, non-label inputs: `impressions_90d`, `avg_position`, and `ctr`. It assigns one score, one reason code, and one action label.

In [2]:
output_path = Path("D:/Flyrank/Flyrank-assignments/work/outputs/baseline_action_score.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

expected_ctr_lookup = (
    visible.groupby("position_bucket", observed=False)
    .agg(total_clicks=("clicks_90d", "sum"), total_impressions=("impressions_90d", "sum"))
)
expected_ctr_lookup["expected_ctr_by_position"] = (
    expected_ctr_lookup["total_clicks"] / expected_ctr_lookup["total_impressions"] * 100
)
expected_ctr_map = expected_ctr_lookup["expected_ctr_by_position"].astype(float).to_dict()

queue = visible.copy()
queue["expected_ctr_by_position"] = queue["position_bucket"].map(expected_ctr_map).astype(float)
queue["ctr_gap"] = (queue["expected_ctr_by_position"] - queue["ctr"]).clip(lower=0)
queue["visibility_rank"] = np.log1p(queue["impressions_90d"]).rank(pct=True)
queue["position_ready"] = queue["avg_position"].between(1, 20).astype(int)
queue["baseline_action_score"] = (
    queue["ctr_gap"] * queue["visibility_rank"] * queue["position_ready"] * 100
).round(3)
queue["reason_code"] = np.where(
    queue["baseline_action_score"] >= 10,
    "low_ctr_vs_position",
    "small_or_no_ctr_gap",
)
queue["action_label"] = np.where(
    queue["baseline_action_score"] >= 10,
    "refresh_and_review_ctr",
    "monitor",
)
queue = queue.sort_values(
    ["baseline_action_score", "impressions_90d", "ctr"],
    ascending=[False, False, True],
).reset_index(drop=True)
queue["baseline_rank"] = np.arange(1, len(queue) + 1)

queue_columns = [
    "baseline_rank",
    "content_id",
    "client_id",
    "baseline_action_score",
    "reason_code",
    "action_label",
    "impressions_90d",
    "avg_position",
    "ctr",
    "expected_ctr_by_position",
    "ctr_gap",
    "position_bucket",
    "days_since_last_update",
    "content_type",
    "main_intent",
]
queue_to_write = queue[queue_columns].copy()
queue_to_write.to_csv(output_path, index=False)

print(f"Wrote ranked queue to: {output_path}")
print("\nAction counts")
print(queue_to_write["action_label"].value_counts().to_string())
print("\nTop 10 rows from the ranked queue")
print(queue_to_write.head(10).to_string(index=False))

Wrote ranked queue to: D:\Flyrank\Flyrank-assignments\work\outputs\baseline_action_score.csv

Action counts
action_label
monitor                   10796
refresh_and_review_ctr     2716

Top 10 rows from the ranked queue
 baseline_rank           content_id         client_id  baseline_action_score         reason_code           action_label  impressions_90d  avg_position  ctr  expected_ctr_by_position  ctr_gap position_bucket  days_since_last_update    content_type   main_intent
             1 content_4a6607efcb46 client_6208ef0f77                 47.778 low_ctr_vs_position refresh_and_review_ctr           128068           2.2 0.01                  0.491239 0.481239           top_3                     104 keyword article informational
             2 content_8451fc6f034d client_d029fa3a95                 46.069 low_ctr_vs_position refresh_and_review_ctr           272144           2.3 0.03                  0.491239 0.461239           top_3                      20 keyword article information

## 3. Top-10 review

One line per row: action, why it is here, confidence note, and what would make it wrong.

In [3]:
top_ten = queue.head(10).copy()


def confidence_note(row):
    if row["baseline_action_score"] >= 35:
        return "high"
    if row["baseline_action_score"] >= 20:
        return "medium"
    return "cautious"


def wrong_if(row):
    if row["days_since_last_update"] <= 14:
        return "A very recent update may not have settled yet, so the low CTR could be temporary."
    if row["main_intent"] == "navigational":
        return "Navigational intent can cap CTR in ways this bucket average does not see."
    if row["content_type"] == "comparison article":
        return "Comparison pages can live in a different SERP pattern than the generic position bucket."
    return "SERP features or intent mismatch could make this page's fair CTR lower than the bucket average."

review_rows = []
for row in top_ten.itertuples(index=False):
    review_rows.append(
        {
            "baseline_rank": row.baseline_rank,
            "content_id": row.content_id,
            "action": row.action_label,
            "why_it_is_here": (
                f"CTR {row.ctr:.2f} vs expected {row.expected_ctr_by_position:.2f} at {row.position_bucket}; "
                f"{int(row.impressions_90d):,} impressions make the gap matter."
            ),
            "confidence_note": confidence_note(row._asdict()),
            "what_would_make_it_wrong": wrong_if(row._asdict()),
        }
    )

top_ten_review = pd.DataFrame(review_rows)
top_ten_review

   baseline_rank  ...                           what_would_make_it_wrong
0              1  ...  SERP features or intent mismatch could make th...
1              2  ...  SERP features or intent mismatch could make th...
2              3  ...  A very recent update may not have settled yet,...
3              4  ...  SERP features or intent mismatch could make th...
4              5  ...  SERP features or intent mismatch could make th...
5              6  ...  SERP features or intent mismatch could make th...
6              7  ...  SERP features or intent mismatch could make th...
7              8  ...  A very recent update may not have settled yet,...
8              9  ...  SERP features or intent mismatch could make th...
9             10  ...  SERP features or intent mismatch could make th...

[10 rows x 6 columns]


## 4. Weak picks and leakage check

A negative signal is useful if it stops a bad rule. This section calls out weak picks and shows the risky inputs I refused to use.

In [4]:
weak_picks = queue.head(25).loc[
    queue.head(25)["days_since_last_update"] <= 14,
    [
        "baseline_rank",
        "content_id",
        "baseline_action_score",
        "days_since_last_update",
        "action_label",
    ],
].copy()
weak_picks["why_watch_it"] = "Recent update: the page may simply need more time before a CTR review is fair."

used_inputs = ["impressions_90d", "avg_position", "ctr"]
excluded_risky_inputs = [
    "trend_direction",
    "trend_pct",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "is_declining_label",
]
leakage_check = pd.DataFrame(
    [
        {
            "rule_uses_only_current_window_inputs": True,
            "used_inputs": ", ".join(used_inputs),
            "excluded_risky_inputs": ", ".join(excluded_risky_inputs),
            "why": "The rule avoids future windows and label-derived columns, so it does not read the answer from trend fields.",
        }
    ]
)

print("Weak picks inside the top 25")
print(weak_picks.to_string(index=False))
print("\nLeakage check")
print(leakage_check.to_string(index=False))

Weak picks inside the top 25
 baseline_rank           content_id  baseline_action_score  days_since_last_update           action_label                                                                   why_watch_it
             3 content_e12868d1f396                 41.937                       7 refresh_and_review_ctr Recent update: the page may simply need more time before a CTR review is fair.
             8 content_954cc45bd437                 37.229                       7 refresh_and_review_ctr Recent update: the page may simply need more time before a CTR review is fair.
            11 content_998f6f88784c                 36.581                       8 refresh_and_review_ctr Recent update: the page may simply need more time before a CTR review is fair.

Leakage check
 rule_uses_only_current_window_inputs                        used_inputs                                                                                                                                              ex

## Self-check

Before you submit, confirm each line honestly:

- [ ] Two signal verdicts are visible, each with a bucket table and n
- [ ] At least one tested signal is linked to a real FlyRank flag idea
- [ ] One rule is encoded with a score, one reason code, and one action label
- [ ] The ranked queue is written to `work/outputs/baseline_action_score.csv`
- [ ] The top 10 each have an action, a why, and what would make the pick wrong
- [ ] No future-window or label-derived inputs were used